## اختبار تأثير تجميع المستندات (Document Clustering)

هذا الـ Notebook ليس لتقييم MAP، بل لإظهار **كيف يمكننا الاستفادة من نتائج التجميع بشكل عملي** لتحسين تجربة المستخدم.

**الفكرة:** بعد إجراء بحث عادي، سنقوم بجلب النتائج الأعلى، ثم لكل نتيجة، سنعرض **5 مستندات أخرى من نفس العنقود (Cluster)**. هذا يعادل ميزة "مستندات أخرى قد تهمك" أو "تصفح هذا الموضوع".

### الخطوة 1: الإعداد وتحميل الملفات اللازمة

In [1]:
import requests
import pandas as pd
import joblib
import sys
import os
import random

project_root = os.path.dirname(os.path.abspath(os.getcwd()))
sys.path.append(project_root)
from config import API_PORTS, MODELS_DIR

# --- الإعدادات ---
DATASET_NAME = 'antique'
SEARCH_API_URL = f"http://127.0.0.1:{API_PORTS['SEARCH']}/search/"
MODEL_DIR = os.path.join(project_root, MODELS_DIR, DATASET_NAME.replace('/', '_'))

try:
    print("Loading required model files...")
    doc_ids = joblib.load(os.path.join(MODEL_DIR, 'doc_ids.joblib'))
    clusters = joblib.load(os.path.join(MODEL_DIR, 'clusters.joblib'))
    
    # إنشاء بنية بيانات فعالة للبحث عن المستندات في كل عنقود
    cluster_to_docs = {}
    for doc_id, cluster_id in zip(doc_ids, clusters):
        if cluster_id not in cluster_to_docs:
            cluster_to_docs[cluster_id] = []
        cluster_to_docs[cluster_id].append(doc_id)
        
    print("Files loaded and cluster map created successfully.")
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure you have run the representation and clustering processes for '{DATASET_NAME}'.")

Loading required model files...
Files loaded and cluster map created successfully.


### الخطوة 2: إجراء بحث وعرض النتائج مع "مستندات ذات صلة"

In [2]:
def search_and_find_related(query, model='hybrid'):
    print(f"--- Searching for: '{query}' ---\n")
    
    # 1. إجراء البحث العادي
    payload = {"dataset_name": DATASET_NAME, "query": query, "model_type": model}
    response = requests.post(SEARCH_API_URL, json=payload)
    top_results = response.json().get('results', [])
    
    if not top_results:
        print("No results found.")
        return
    
    # 2. لكل نتيجة، جلب مستندات أخرى من نفس العنقود
    for i, result in enumerate(top_results[:3]): # نعرض أول 3 نتائج فقط للتوضيح
        doc_id = result['doc_id']
        score = result['score']
        
        # إيجاد العنقود الذي ينتمي إليه هذا المستند
        doc_index = doc_ids.index(doc_id)
        cluster_id = clusters[doc_index]
        
        # جلب مستندات أخرى من نفس العنقود
        related_docs = cluster_to_docs.get(cluster_id, [])
        # إزالة المستند نفسه من قائمة المستندات ذات الصلة
        related_docs = [d for d in related_docs if d != doc_id]
        # أخذ عينة عشوائية
        sample_related = random.sample(related_docs, min(len(related_docs), 5))
        
        print(f"\n===================================================")
        print(f"Result #{i+1} | Doc ID: {doc_id} | Score: {score:.4f} | Cluster: {cluster_id}")
        print(f"---------------------------------------------------")
        print(f"-> Other documents in the same topic cluster:")
        for related_id in sample_related:
            print(f"   - Doc ID: {related_id}")
        print(f"===================================================")

# --- تجربة عملية ---
test_query = "what is the best way to learn python?"
search_and_find_related(test_query)

--- Searching for: 'what is the best way to learn python?' ---


Result #1 | Doc ID: 4179739_0 | Score: 0.8000 | Cluster: 10
---------------------------------------------------
-> Other documents in the same topic cluster:
   - Doc ID: 2879709_0
   - Doc ID: 692417_2
   - Doc ID: 4126337_4
   - Doc ID: 2571027_8
   - Doc ID: 118868_1

Result #2 | Doc ID: 2577771_0 | Score: 0.5295 | Cluster: 2
---------------------------------------------------
-> Other documents in the same topic cluster:
   - Doc ID: 1217838_1
   - Doc ID: 3978007_3
   - Doc ID: 147624_12
   - Doc ID: 389744_12
   - Doc ID: 881894_1

Result #3 | Doc ID: 2834963_3 | Score: 0.3867 | Cluster: 6
---------------------------------------------------
-> Other documents in the same topic cluster:
   - Doc ID: 3959752_4
   - Doc ID: 1381517_2
   - Doc ID: 648371_2
   - Doc ID: 3411998_6
   - Doc ID: 2778723_16
